# Phase 2 — Imbalance and bounded candidates

Run `python scripts/phase_02_acceptance.py` first. SMOTE lives inside an imblearn Pipeline. It is applied only during training, after train-fitted scaling. The frozen minority/majority ratio is 0.1 (not full balancing). Weighted XGBoost uses the training negative/positive ratio and six fixed CPU configurations. Isolation Forest is an unsupervised comparison; risk is negative score_samples so high means anomalous.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from fraud_detection.paths import find_project_root
from fraud_detection.pipelines.train import load_verified_results

ROOT = find_project_root()
OUTPUT = ROOT / 'artifacts/phase_02'
result = load_verified_results(ROOT)
print('Source kind:', result['source_kind'])
print('Config SHA:', result['config_sha256'])
print('Test opened:', result['test_opened'])


In [ ]:
display(pd.DataFrame(result['comparison']).sort_values('ap_val', ascending=False))
rows = []
for run in result['runs']:
    rows.append({'model': run['model_name'], 'train_before': run['train_class_counts_before'],
                 'train_after': run['train_class_counts_after'],
                 'validation': run['validation_class_counts'],
                 'scaler_fit_rows': run['scaler_fit_rows'],
                 'fit_source': run['fit_source'], 'test_opened': run['test_opened']})
display(pd.DataFrame(rows))
table = pd.DataFrame(result['comparison']).set_index('model')
table['ap_val'].plot.bar(title='Primary validation AP by fixed candidate', ylabel='Average precision')
plt.tight_layout()
plt.show()
reference = json.loads((OUTPUT / 'reference/experiment.json').read_text())
print('Independent reference split SHA:', reference['split_sha256'])
display(pd.DataFrame(reference['comparison']))


## Limits

The reference split is an independent protocol; its scores do not enter primary candidate selection. Six XGBoost settings are an intentionally bounded search. Repeated validation comparisons create selection bias; the proposal still requires peer review before test is opened. No model binary is published.